This notebook prepares the directory structure and parameter files required for the LC-MS/MS data analysis pipeline.  
It reads the raw data exported from SCIEX Analyst or Waters Connect in your project folder, identifies compounds, standards, and method types, and automatically generates all input files used by `data_analysis.ipynb`.

You will specify where your project folder is located, define the identifiers used in your dataset (EIS, NIS, HRMS), the instrument (waters, sciex) and then run the notebook.  
After execution, the notebook creates standardized parameter files that you can edit to reflect experiment-specific settings such as recovery thresholds, RT/IAR criteria, sample metadata, precalculated method detection limits, and simulation options.

Once the templates are customized, your project folder is ready for the full analysis in `data_analysis.ipynb`.

# Step by step instructions

This section explains how to prepare your project directory, set naming conventions, and adjust the configuration variables in the notebook before running the workflow.

---

## 1. Prepare Your Project Folder
Before running the notebook, create a dedicated folder for your project and place all raw data files inside it.
These should be the exported .csv or .txt results from Sciex Analyst or Waters Connect, with *all columns included*.
**The project folder should be located in the same parent folder as this script.**

To ensure correct processing, follow these rules:
1. All files from Waters stem from the same method, per defaults files from Waters must end with: `_core.txt` or `_core.csv`.
2. For Sciex, multiple methods exist.
- Data generated using the **core method** must end with: `_core.txt` or `_core.csv`
- Data generated using the **extended method** must end with: `_extended.txt` or `_extended.csv`
- If you want to combine results from core and extended method from Sciex, the **base filename** (the part *before* `_core` or `_extended`) must be **identical** across the paired files.


## 2. Set `project_folder` in the Notebook
Update the variable `project_folder` in the first code block so it points to the project directory you just created.

- **Windows:** Right-click the folder → *Copy as path*  
- **macOS:** Control/right-click folder → hold **Option (Alt)** → *Copy “[foldername]” as Pathname*

Paste the copied path into the notebook variable.

## 3. Adjust Identifier Variables
Update the variables that specify how the notebook recognizes different types of compounds:

### Instrument identifier
Set `data_format` to match your instrument:
- **Sciex:** `sciex`  
- **Waters:** `waters`

### Extracted and Non-Extracted Internal Standards
Set:
- `eis_identifier` (Extracted Internal Standard)
- `nis_identifier` (Non-Extracted Internal Standard)

Use the correct identifiers depending on calibration date:
- **Before June 2025:** `IDA` (EIS) and `IPS` (NIS)  
- **After June 2025:** `EIS` and `NIS`

### HRMS Channel Identifier
Set `hrms_identifier` to match the channel naming convention in your data:
- **Sciex before June 2025:** `_TOF MS`  
- **Sciex after June 2025:** `_HRMS`
- **Waters (starting August 2026):** `_Qual`

## 4. Run the Notebook
After running all cells of this jupyter notebook, two new directories will appear inside your project folder:

1. **`code_parameters`**  
   Contains five CSV files used to configure thresholds and metadata:  
   - `eis_parameters.csv`
   - `compound_parameters.csv`
   - `sample_parameters.csv`
   - `simulation_parameters.csv`
   - `mdl_parameters.csv` 

2. **`processed_data`**  
   Contains the subdirectory:  
   - `plots`

Ensure both folders were successfully created.

## 5. Update Code Parameter Files

### (i) EIS parameters  
Open `eis_parameters.csv` and set thresholds appropriate for your study.  

For ultra-short-chain PFAS (where no NIS is available), set **method to calculate recoveries** in `simulation_parameters.csv` to `compound` (see Section 5 (v)) to evaluate **internal standard responses** instead of **recovery rates**.

Save and close the file.

### (ii) Compound Parameters  
In `compound_parameters.csv`:
- Adjust **ion abundance ratio thresholds**  
- In the **channel selection** column, set the value to `HRMS` if quantification should use the HRMS channel (works for Sciex only), or to `NONE` if the compound should be excluded from the final results.
Save and close the file.


### (iii) MDL Parameters
In `mdl_parameters.csv`:
- To calculate MDLs from your current data file, leave this file unchanged. This is the default behavior, controlled by the `method to calculate mdl` field in `simulation_parameters.csv`.
- To use precalculated MDLs, enter your values in `mdl_parameters.csv` and set `method to calculate mdl` in `simulation_parameters.csv` to `manually`.

### (iv) Sample Parameters  
In `sample_parameters.csv`:
- Update sample names (column: *alternative name (used in results)*).  
- Do **not** change *sample number* or *sample ID*.  
- Provide:
  - sample size (`volume, weight, number of samples, etc.`)  
  - units (`g/mL/sample`)  
- If you set method to calculate mdl to `code` in `simulation_parameters.csv` (see Section 5 (v)), mark the blanks that should be used for MDL calculation by setting `used for mdl calculation` = `TRUE`. 
- For passive samplers, enter:
  - temperature (°C)  
  - deployment time (days)  
- Update dilution factors if necessary.  
Save and close the file.

### (v) Simulation Parameters  
In `simulation_parameters.csv`:
- Enter an `output name` of your choice. All output files from this run will be saved using this name.
- `calibration reference`: Decide which calibration data should be used as reference for QA/QC. Set to `calibration` to use the initial calibration, or `ccv` to use the continuous calibration verification.
- Change the `method identifier` to match your data format. This variable is used to read in appropriate corresponding instrumentation detection limits from the lab_parameters folder within this github project.
  - Recommended for old SCIEX data (before July 2025, when the high resolution channel was named TOF MS): `2024_test_anonymous`
  - Recommended for new SCIEX data (after July 2025, when the high resolution channel was named HRMS): `2025_water_anonymous`
  - Recommended for Waters data: `2026_waters_simon`.
  - If you developed your own method and want to use your own mdl values, run the `calculated_idl.py` script using your data files as inputs and set the name of the created idl data file to the `method identifier` in your `simulation_parameters.csv`.
- Indicate how your standards were spiked. Set to `spiked_per_sample` if a fixed amount was added per sample (typical GSO procedure), or `spiked_per_ml_sample` if the amount was normalized to sample volume (typical COP procedure).
- `method to calculate recoveries`: Choose how recoveries should be calculated. Set to `eis` to use EIS and their assigned NIS for recovery calculation (standard procedure), or `compound` in case you only spiked EIS and calculate recoveries based on EIS areas in the current sample compared to EIS average areas in defined calibration data. (See also (i))
- `method to calculate mdl`: Choose how method detection limits should be handled. Set to `manually` to use the values you provide in `mdl_parameters.csv`, or `code` to have the code calculate MDLs from your current data file. (See also (iii)).
- `method to calculate ion abundance ratios`: Choose the basis for calculating ion abundance ratio deviation (IARD). Set to `area` to use peak areas, or `concentration` to use concentrations. For Waters data only `area` is feasible.
- `use B5DL flag`: Choose whether to additionally flag concentration values that are below 5 times the method detection limit (MDL). Set to `TRUE` to flag such values as `B5DL` (in addition to the standard `BDL` flag for values below the MDL), or `FALSE` to apply only the standard `BDL` flag.

## 6. Run `data_analysis.ipynb`
After adjusting the parameter files, run the notebook `data_analysis.ipynb` to process your data and generate results. This is a different jupyter notebook provided in this github project.

In [ ]:
# relevant input variables
project_folder = r'test/waters'  # path to the project folder
data_format = 'waters'  # 'waters' or 'sciex'

# used to identify extracted internal standards (EIS), previously known as IDA, 
# non-extracted internal standards (NIS) previously known as IPS, and
# high resolution mass spectrometry (_HRMS) channel, previously known as _TOF MS
eis_identifier = 'EIS'
nis_identifier = 'NIS'
hrms_identifier = "_Qual"

# used to identify internal standards from column "Component Name" - usually you do not have to change it
standard_identifiers = 'Avg|EIS|NIS|IDA|IPS|13C|d-|d3-|d5-|18O'

In [ ]:
# import necessary packages
import pandas as pd
import os

# import functions from utils.py
from utils import (
    read_in_data_files, get_sample_id_and_name, clean_up_data, get_hrms_and_msms_compounds, get_hrms_and_msms_standards,
    )

# display settings
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_colwidth', None)  # Show full width of columns

In [ ]:
# create directory for output data, thow error if it already exists
if os.path.isdir(os.path.join(project_folder, 'processed_data')):
    raise ImportError("""
        Processed Data Folder already exists in your project folder.
        The script does not create a new one.            
                      """)
os.makedirs(os.path.join(project_folder, 'processed_data'))
os.makedirs(os.path.join(project_folder, 'processed_data', 'plots'))

In [ ]:
# create directory for code parameters, thow error if it already exists
if os.path.isdir(os.path.join(project_folder, 'code_parameters')):
    raise ImportError("""
        Simulation Parameters Folder already exists in your project folder.
        The script does not create a new one.            
                      """)
os.makedirs(os.path.join(project_folder, 'code_parameters'))

In [ ]:
### Create csv file sample_parameters.csv for sample inputs.
# Information about available samples, LCMS Code and internal names is extracted from raw data.

# combine all data files to one common dataframe
data, output_name = read_in_data_files(project_folder, hrms_identifier=hrms_identifier, data_format=data_format)

# extract list of samples from raw data
sample_list = get_sample_id_and_name(data)
display(sample_list)

# for simplicity: get rid of standard samples in sample list
visible_sample_list = sample_list.loc[sample_list['Sample Type'] != 'Standard', :]

# create a sample input parameter file
sample_input_data = pd.DataFrame(columns=[
    'batch name', 'sample id', 'alternative name (used in results)', 
    'volume/weight/number of samples', 'unit (e.g. g/mL/sample)', 'used for mdl calculation',
    'temperature in C', 'deployment time in days', 'dilution factor',
    ],
    index=visible_sample_list.index
    )
# fill sample id column
sample_input_data['sample id'] = visible_sample_list['Sample ID']
sample_input_data['batch name'] = visible_sample_list['Batch Name']
# fill sample names column and get rid of 'Core' and 'Ext' ending of sample names, as Core and Extended method will be combined by the code internally.
sample_names = []
for (sample_name_core, sample_name_extended) in zip(visible_sample_list['Sample Name Core'], visible_sample_list['Sample Name Extended']):
    if pd.isnull(sample_name_extended):
        sample_names.append(sample_name_core[:-5])
    else:
        sample_names.append(sample_name_extended[:-4])
sample_input_data['alternative name (used in results)'] = sample_names

# set default values for remaining columns
sample_input_data['volume/weight/number of samples'] = 1
sample_input_data['used for mdl calculation'] = False
sample_input_data['unit (e.g. g/mL/sample)'] = 'sample'
sample_input_data['dilution factor'] = 1

# write dataframe to csv
sample_input_data.to_csv(path_or_buf=str(os.path.join(project_folder, 'code_parameters', 'sample_parameters.csv')), index=True)

In [ ]:
# Information about available target analytes and standards is extracted from raw data.

# clean up data first (misleading compound names are corrected)
data = clean_up_data(data=data, sample_list=sample_list)

# extract list of compound names in right order from raw data
compounds, _, _ = get_hrms_and_msms_compounds(
    data=data, sample_list=sample_list, hrms_identifier=hrms_identifier, standard_identifiers=standard_identifiers,
    )
display(compounds)

# extract list of standard names in right order from raw data
standards, _ = get_hrms_and_msms_standards(
    data=data, sample_list=sample_list, hrms_identifier=hrms_identifier, standard_identifiers=standard_identifiers,
    eis_identifier=eis_identifier, nis_identifier=nis_identifier, compound_order_offset=int(compounds['Compound Order'].max())
    )
display(standards)

In [ ]:
### Create csv file eis_parameters.csv for threshold inputs for recovery rates or internal standard response deviations.
eis_sorted = standards.loc[standards['Standard Type'] == eis_identifier, 'MSMS Standard Name'].to_list()
nis_sorted = standards.loc[standards['Standard Type'] == nis_identifier, 'MSMS Standard Name'].to_list()

# create default data frame for thresholds
recovery_or_standard_response_thresholds = pd.DataFrame(columns=[
    'lower threshold [%]', 'upper threshold [%]'
    ],
    index=eis_sorted
    )
# input default thresholds
recovery_or_standard_response_thresholds['lower threshold [%]'] = 50
recovery_or_standard_response_thresholds['upper threshold [%]'] = 150
# write data frame to csv
recovery_or_standard_response_thresholds.to_csv(path_or_buf=str(os.path.join(project_folder, 'code_parameters', 'eis_parameters.csv')), index=True)

In [ ]:
### Create csv file compound_parameters.csv for 
# threshold inputs for accepted retention time differences and 
# accepted ion abundance ratio deviations
# as well as the selection of channels used for quantification

hrms_label = str.upper(hrms_identifier.replace('_', ''))  # get column label for HRMS channel

# create default data frame for accepted retention time differences and ion abundance ratio deviations
rt_iar_thresholds_channel_selection = pd.DataFrame(columns=[
    'accepted retention time difference [min]', 'accepted ion abundance ratio deviation [%]', hrms_label + ' Standard Name',
    'MDL MS/MS [ng/sample]', 'MDL' + hrms_label + ' [ng/sample]',
    ],
    index=compounds['MSMS Compound Name'].fillna(compounds[hrms_label + ' Compound Name']).tolist()
    )
# input default thresholds
rt_iar_thresholds_channel_selection['accepted retention time difference [min]'] = 0.1
rt_iar_thresholds_channel_selection['accepted ion abundance ratio deviation [%]'] = 50
rt_iar_thresholds_channel_selection['Selected channel for quantification [MSMS/' + hrms_label + '/NONE]'] = 'MSMS'

mask_nones = compounds['MSMS Compound Name'].isna() | compounds[hrms_label + ' Compound Name'].isna()

rt_iar_thresholds_channel_selection.loc[mask_nones.tolist(), 'Selected channel for quantification [MSMS/' + hrms_label + '/NONE]'] = 'NONE'

# set threshold to 0.4 minutes if corresponding EIS has different chemical structure
# loop over pfas compounds
for pfas_compound in rt_iar_thresholds_channel_selection.index.tolist():
    # skip standards
    if eis_identifier in pfas_compound or nis_identifier in pfas_compound:
        continue
    # check if any EIS matches names of PFAS compound -> means exact chemical structure is available
    # and skip iteration of loop if it is available
    if len([1 for eis in eis_sorted if pfas_compound in eis]):
        continue
    # set the retention time threshold to 0.4 minutes if exact EIS name was not available.
    rt_iar_thresholds_channel_selection.loc[pfas_compound, 'accepted retention time difference [min]'] = 0.4

rt_iar_thresholds_channel_selection.to_csv(path_or_buf=str(os.path.join(project_folder, 'code_parameters', 'compound_parameters.csv')), index=True)

In [ ]:
### Create csv file simulation_parameters.csv for inputs to the data_analysis.ipynb notebook.
# create default data frame for recovery thresholds
simulation_parameters = pd.DataFrame(columns=['Parameter Name', 'Parameter Description', 'Parameter Value'])

# Write parameters, descriptions and default values to data frame.
simulation_parameters.loc[0] = [
"EIS identifier", "Repeating substring, which is used to identify extracted internal standards (EIS), formally known as IDA, from compound name.", eis_identifier,
]
simulation_parameters.loc[1] = [
"NIS identifier", "Repeating substring, which is used to identify non-extracted internal standards (NIS), formally known as IPS, from compound name.", nis_identifier,
]

simulation_parameters.loc[2] = [
"data format", "Format of the data files (sciex or waters)", data_format,
]

simulation_parameters.loc[3] = [
"HRMS identifier", "Ending of compound names (column Component Name in SCIEX) for compounds and standards from the HRMS channel.", hrms_identifier,
]


simulation_parameters.loc[4] = [
"calibration midpoint identifier", "Repeating substring, which is used to identify calibration midpoint from sample name.", "CS6"
]

simulation_parameters.loc[5] = [
"output name", "Name of the output files generated from the Code", output_name 
]

simulation_parameters.loc[6] = [
"calibration reference", "To choose reference data from the calibration (midpoint) for QAQC, you can use data from initial calibration, or data from continuous calibration verification (CCV) \n"+ \
"Set parameter to calibration if you want to use data from initial calibration as reference. \n" +\
"Set parameter to ccv if you want to use data from continuos calibration verification as reference.", "calibration"
]

simulation_parameters.loc[7] = [
"method identifier", "{Year of method creation}_{matrix type}_{name of researcher who created the method}", "2024_test_anonymous"
]

simulation_parameters.loc[8] = [
"standard concentration", "Available options: 'spiked_per_sample', 'spiked_per_ml_sample'.", "spiked_per_sample"
]

simulation_parameters.loc[9] = [
"method to calculate recoveries", "Available options: 'eis', 'compound'.", "eis"]

simulation_parameters.loc[10] = [
"method to calculate mdl", "Available options: 'manually': if you want to input mdl files manually via (mdl_parameters.csv), \n" + \
"or 'code': if mdls are calculated by the code.", "code"]

simulation_parameters.loc[11] = [
"method to calculate ion abundance ratios", "Available options: 'area': if you want to calculate IARD based on peak areas, \n" + \
"or 'concentration': if you want to calculate IARD based on concentrations.", "area"]

simulation_parameters.loc[12] = [
"use B5DL flag", "Whether to flag concentration values that are below 5 times the method detection limit (MDL). \n" + \
"Available options: TRUE or FALSE. \n" + \
"If TRUE, values below 5x MDL are flagged as 'B5DL' (in addition to the standard BDL flag for values below MDL). \n" + \
"If FALSE, only the standard BDL flag is applied.", False
]

# set parameter to index and delete column
simulation_parameters.set_index('Parameter Name', inplace=True)

# write data frame to csv
simulation_parameters.to_csv(path_or_buf=str(os.path.join(project_folder, 'code_parameters', 'simulation_parameters.csv')), index=True)

In [ ]:
# create file to input MDLs
mdl = pd.DataFrame(
    columns=['MDL MS/MS [ng/sample]', 'MDL' + hrms_label + ' [ng/sample]'],
    index=compounds['MSMS Compound Name'].fillna(compounds[hrms_label + ' Compound Name']).tolist()
    )
mdl[:] = 1e-3
mdl.to_csv(path_or_buf=str(os.path.join(project_folder, 'code_parameters', 'mdl_parameters.csv')), index=True)